# Spine MRI Analysis Pipeline — Thoracic

End-to-end inference pipeline for thoracic spine MRI on Google Colab (H100 / A100, 80 GB recommended).

**Patient case** *(анонимно)*: бывший спортсмен, контралатеральный паттерн боли (наклон влево → боль справа) → подозрение на дисфункцию рёберно-позвоночных / фасеточных суставов. Радиолог не нашёл значимых изменений. Задача AI — выявить тонкие находки.

**Tools used** *(per Gemini Deep Research, 2026-05-21)*:
| Stage | Tool | Targets |
|-------|------|---------|
| Conversion | `dcm2niix` | DICOM → NIfTI |
| Segmentation A | `SPINEPS` | Тела позвонков + задние элементы (фасетки, отростки) |
| Segmentation B | `TotalSpineSeg` | Спинной мозг, канал, диски |
| Segmentation C | `TotalSegmentator MRI v2` | Параспинальные мышцы, рёбра |
| Anomaly detection | `U2AD` | T2-гиперинтенсивности (BME, отёк) |
| Disc grading | `SpineNetV2` | Pfirrmann/Modic (относительный ранг) |
| Texture | `pyradiomics` | GLCM/GLRLM на ROI |
| Geometry | custom | Cobb, wedging, kyphosis из центроидов |

**Output**: `results/findings.json` + `results/findings.csv` (structured metrics).

**Runtime**: GPU required. ~20–30 min on H100.

**Robustness**: каждая ячейка обёрнута в try/except. Если инструмент падает (например, веса недоступны) — соответствующее поле в финальном JSON = `null` с указанием причины. Pipeline не останавливается.

⚠️ **Дисклеймер**: вывод носит исследовательский характер, не заменяет квалифицированного радиолога.

## Cell 1 — Setup

In [ ]:
# === Cell 1a: Imports, GPU check, workspace + torch.load in-kernel patch ===
import os, sys, json, time, subprocess, shutil, traceback
from pathlib import Path
import numpy as np

import torch

# ─── PATCH torch.load directly in this kernel process ──
# sitecustomize.py is unreliable in the Colab IPython kernel (kernel skips
# site processing). Apply the patch explicitly here so every tool that runs
# in-process (TotalSegmentator Python API, SPINEPS Python API, etc.) sees
# weights_only=False.
if not getattr(torch.load, '__wrapped_torch_load__', False):
    _orig_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _orig_torch_load(*args, **kwargs)
    _patched_torch_load.__wrapped_torch_load__ = True
    torch.load = _patched_torch_load
    print("✓ torch.load patched in-kernel")
else:
    print("✓ torch.load already patched")

# Belt-and-suspenders: register safe globals so even if a tool unwraps
# torch.load, weights_only=True still works for numpy-containing checkpoints
try:
    torch.serialization.add_safe_globals([
        np.core.multiarray.scalar,
        np.ndarray,
        np.dtype,
    ])
    print("✓ torch safe globals registered")
except Exception as e:
    print(f"  [info] add_safe_globals not applied: {e}")

assert torch.cuda.is_available(), "Switch runtime to GPU (Runtime → Change runtime type → H100/A100)"
print(f"\nGPU:    {torch.cuda.get_device_name(0)}")
print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA:   {torch.version.cuda}")
print(f"Torch:  {torch.__version__}")

WORK         = Path('/content/spine_work')
DICOM_DIR    = WORK / 'data' / 'dicom'
NIFTI_DIR    = WORK / 'data' / 'nifti'
RESULTS_DIR  = WORK / 'results'
INTERMEDIATE = RESULTS_DIR / 'intermediate'
DEBUG_DIR    = RESULTS_DIR / 'debug'
WEIGHTS_DIR  = WORK / 'weights'
REPO_DIR     = WORK / 'repo'
EXT_DIR      = WORK / 'ext'

for d in [WORK, DICOM_DIR, NIFTI_DIR, RESULTS_DIR, INTERMEDIATE, DEBUG_DIR, WEIGHTS_DIR, REPO_DIR.parent, EXT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STATUS = {'tools': {}}

def mark(tool, status, reason=None, extra=None):
    STATUS['tools'][tool] = {'status': status, 'reason': reason}
    if extra:
        STATUS['tools'][tool].update(extra)
    print(f"[{tool:20s}] {status}" + (f" — {reason}" if reason else ""))

print(f"\nWorkspace: {WORK}")


In [ ]:
# === Cell 1b: System packages ===
!apt-get -qq update > /dev/null
!apt-get -qq install -y dcm2niix unzip git > /dev/null
print("apt packages: dcm2niix, unzip, git OK")

In [ ]:
# === Cell 1c: Robust environment setup (v8) ===
# v8 adds two installs to fix the remaining tool failures from v7 run:
#   - kornia<0.8 (kornia 0.8+ removed Tensor from kornia.core, breaking TotalSpineSeg)
#   - cupy-cuda12x (TotalSegmentator imports CuPy; we run on CUDA 12.8)

import os, subprocess, sys

SETUP_MARKER = '/tmp/_spine_pipeline_setup_v8'
SC_PATH      = '/usr/local/lib/python3.12/dist-packages/sitecustomize.py'

PATCH_MARKER = '# === spine_pipeline torch.load patch ==='
PATCH_BLOCK = (
    f"\n{PATCH_MARKER}\n"
    "try:\n"
    "    import torch, numpy\n"
    "    try:\n"
    "        torch.serialization.add_safe_globals([\n"
    "            numpy.core.multiarray.scalar,\n"
    "            numpy.ndarray,\n"
    "            numpy.dtype,\n"
    "        ])\n"
    "    except Exception:\n"
    "        pass\n"
    "    _orig = torch.load\n"
    "    def _patched(*args, **kwargs):\n"
    "        kwargs['weights_only'] = False\n"
    "        return _orig(*args, **kwargs)\n"
    "    _patched.__wrapped_torch_load__ = True\n"
    "    torch.load = _patched\n"
    "except Exception:\n"
    "    pass\n"
)

with open(SC_PATH, 'w') as f:
    f.write(f"# Auto-written by spine_analysis_pipeline.ipynb (Cell 1c v8)\n{PATCH_BLOCK}")
print(f"  wrote {SC_PATH}")

SYS_SC = '/usr/lib/python3.12/sitecustomize.py'
try:
    existing = open(SYS_SC).read() if os.path.exists(SYS_SC) else ""
    if PATCH_MARKER not in existing:
        with open(SYS_SC, 'a') as f:
            f.write(PATCH_BLOCK)
        print(f"  appended patch to {SYS_SC}")
    else:
        print(f"  {SYS_SC} already contains patch")
except PermissionError as e:
    print(f"  [WARN] could not write {SYS_SC}: {e}")

try:
    import site
    user_site = site.getusersitepackages()
    os.makedirs(user_site, exist_ok=True)
    uc_path = os.path.join(user_site, 'usercustomize.py')
    with open(uc_path, 'w') as f:
        f.write(f"# Auto-written by spine_analysis_pipeline (backup patch)\n{PATCH_BLOCK}")
    print(f"  wrote {uc_path}")
except Exception as e:
    print(f"  [info] usercustomize.py not written: {e}")


def pip_one(pkg, upgrade=False, force=False, no_deps=False, label=None):
    args = []
    if force:        args += ['--upgrade', '--force-reinstall']
    elif upgrade:    args += ['--upgrade']
    if no_deps:      args += ['--no-deps']
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + args + [pkg],
                       capture_output=True, text=True)
    name = label or pkg
    if r.returncode == 0:
        print(f"    ✓ {name}")
        return True
    print(f"    ✗ {name}")
    last = r.stderr.strip().split('\n')[-2:]
    for ln in last:
        print(f"        {ln[:200]}")
    return False

if os.path.exists(SETUP_MARKER):
    print("✓ Setup already complete (post-restart)")
else:
    print("\nFirst-time setup (~6-8 min, individual installs)\n")

    print("[1/8] medical I/O:")
    for pkg in ['nibabel', 'pydicom', 'SimpleITK']:
        pip_one(pkg)

    print("\n[2/8] pyradiomics (optional, has skimage fallback):")
    pip_one('pyradiomics', label='pyradiomics (may fail on py3.12, OK)')

    print("\n[3/8] MONAI upgrade (fix custom_warning_handler bug):")
    pip_one('monai', upgrade=True)

    print("\n[4/8] nnunetv2 upgrade FIRST (TotalSegmentator picks it up next):")
    pip_one('nnunetv2', upgrade=True)

    print("\n[5/8] spine tools:")
    pip_one('TotalSegmentator', upgrade=True)
    pip_one('totalspineseg')
    pip_one('spineps')

    print("\n[6/8] kornia pinned for TotalSpineSeg (0.8+ dropped Tensor from kornia.core):")
    pip_one('kornia<0.8', force=True, label='kornia<0.8 (force-downgrade)')

    print("\n[7/8] CuPy for CUDA 12.x (TotalSegmentator needs it):")
    pip_one('cupy-cuda12x', label='cupy-cuda12x')

    print("\n[8/8] post-install fixes:")
    pip_one('acvl_utils', force=True, no_deps=True, label='acvl_utils (force-fix)')
    pip_one('nnunetv2==2.7.0', force=True, no_deps=True,
            label='nnunetv2==2.7.0 (force-pin for TotalSegmentator data_loader)')

    test = subprocess.run(
        [sys.executable, '-c', 'import torchvision; print(torchvision.__version__)'],
        capture_output=True, text=True)
    if test.returncode == 0:
        print(f"    ✓ torchvision {test.stdout.strip()} (left alone)")
    else:
        print(f"    ⚠ torchvision broken; reinstalling")
        pip_one('torchvision', force=True, label='torchvision matched')

    open(SETUP_MARKER, 'w').close()
    print("\n⚠ Restarting kernel to reload everything fresh...")
    print("   After restart: Runtime → Run all (Cell 1c will skip reinstall).")
    os.kill(os.getpid(), 9)


In [ ]:
# === Cell 1d: Environment verification (v7) ===
import importlib, importlib.util, sys, os, subprocess

print("Environment diagnostic:\n")
critical = []

def check(name, attr=None, label=None):
    label = label or name
    try:
        m = importlib.import_module(name)
        ver = getattr(m, '__version__', '?')
        ok = hasattr(m, attr) if attr else True
        icon = '✓' if ok else '⚠'
        suffix = f" ({attr} {'present' if ok else 'MISSING'})" if attr else ''
        print(f"  {icon} {label:38s} {ver}{suffix}")
        return ok
    except Exception as e:
        print(f"  ✗ {label:38s} NOT IMPORTABLE — {str(e)[:120]}")
        return False

check('torch'); check('numpy'); check('pandas')
check('nibabel'); check('pydicom'); check('SimpleITK')
check('scipy'); check('skimage')

# torch.load IN-KERNEL patch (Cell 1a)
import torch
patched = getattr(torch.load, '__wrapped_torch_load__', False)
icon = '✓' if patched else '✗'
print(f"  {icon} torch.load in-kernel override          {'active' if patched else 'NOT ACTIVE'}")
if not patched:
    critical.append('Cell 1a in-kernel patch did not apply — re-run Cell 1a explicitly')

# torch.load SUBPROCESS patch — proves sitecustomize change reaches new Python processes
test = subprocess.run(
    [sys.executable, '-c',
     'import torch; '
     'print("PATCHED" if getattr(torch.load, "__wrapped_torch_load__", False) else "UNPATCHED")'],
    capture_output=True, text=True)
sp_state = (test.stdout or test.stderr).strip()
if 'PATCHED' in sp_state:
    print(f"  ✓ torch.load in subprocess                  active (sitecustomize works)")
else:
    print(f"  ✗ torch.load in subprocess                  {sp_state[:120]}")
    critical.append('subprocess sitecustomize patch missing — TotalSpineSeg CLI will fail')

# sitecustomize trace
print("\n  [sitecustomize debug:]")
try:
    spec = importlib.util.find_spec('sitecustomize')
    if spec is None:
        print(f"    sitecustomize not findable")
    else:
        print(f"    sitecustomize at: {spec.origin}")
        try:
            content = open(spec.origin).read()
            has_marker = '# === spine_pipeline torch.load patch ===' in content
            print(f"    contains our patch marker: {has_marker}")
        except Exception as e:
            print(f"    could not read: {e}")
except Exception as e:
    print(f"    probe failed: {e}")

# torch.load round-trip (cheap sanity)
try:
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
        tmp = f.name
    torch.save({'x': torch.tensor([1.0])}, tmp)
    _ = torch.load(tmp)
    os.unlink(tmp)
    print(f"\n  ✓ torch.load round-trip                     OK")
except Exception as e:
    critical.append(f'torch.load round-trip failed: {e}')

# torchvision
try:
    import torchvision
    print(f"  ✓ torchvision                              {torchvision.__version__}")
except Exception as e:
    critical.append(f'torchvision: {str(e)[:200]}')

check('monai')

# Critical: nnunetv2 data_loader module (TotalSegmentator imports from this)
try:
    from nnunetv2.training.dataloading.data_loader import nnUNetDataLoader  # noqa
    print(f"  ✓ nnunetv2.training.dataloading.data_loader present (nnUNetDataLoader importable)")
except Exception as e:
    critical.append(f'nnunetv2.training.dataloading.data_loader: {e}')
    print(f"  ✗ nnunetv2.training.dataloading.data_loader   {str(e)[:120]}")

check('radiomics', label='radiomics (pyradiomics, optional)')
check('totalsegmentator')
check('totalspineseg')
check('spineps')

try:
    from acvl_utils.cropping_and_padding.bounding_boxes import crop_and_pad_nd
    print(f"  ✓ acvl_utils.crop_and_pad_nd               present")
except ImportError as e:
    critical.append(f'acvl_utils.crop_and_pad_nd: {e}')

if critical:
    print("\n" + "=" * 60)
    print("⚠ CRITICAL FAILURES")
    print("=" * 60)
    for f in critical:
        print(f"  - {f}")
    print("\n  Remediation: Runtime → Disconnect and delete runtime,")
    print("  then File → Revert to GitHub, then Run all from scratch.")
else:
    print("\n✓ Environment ready — running pipeline")


## Cell 2 — DICOM → NIfTI + sequence detection

In [ ]:
# === Cell 2a: Acquire data ===
REPO_URL = "https://github.com/omarnuri/MRI-reseqrch.git"

if not (REPO_DIR / '.git').exists():
    print(f"Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repo already present")

zip_candidates = list(REPO_DIR.glob('*.zip'))
if not zip_candidates:
    raise FileNotFoundError(f"No .zip in {REPO_DIR}. Place DICOM archive there or upload manually.")
DICOM_ZIP = zip_candidates[0]
print(f"DICOM archive: {DICOM_ZIP.name} ({DICOM_ZIP.stat().st_size / 1e6:.1f} MB)")

# Extract if not already done
needs_extract = not any(DICOM_DIR.rglob('*'))
if needs_extract:
    print("Extracting...")
    subprocess.run(['unzip', '-q', '-o', str(DICOM_ZIP), '-d', str(DICOM_DIR)], check=True)

# Sanity-count DICOM files
import pydicom
dcm_files = []
for p in DICOM_DIR.rglob('*'):
    if not p.is_file() or p.suffix.lower() == '.zip':
        continue
    try:
        pydicom.dcmread(str(p), stop_before_pixels=True)
        dcm_files.append(p)
    except Exception:
        pass
print(f"Valid DICOM files: {len(dcm_files)}")

In [ ]:
# === Cell 2b: dcm2niix conversion ===
print("Running dcm2niix...")
res = subprocess.run(
    ['dcm2niix', '-z', 'y', '-f', '%p_%s', '-o', str(NIFTI_DIR), str(DICOM_DIR)],
    capture_output=True, text=True
)
if res.returncode != 0:
    print("dcm2niix STDERR:")
    print(res.stderr[-2000:])
print(res.stdout[-2000:])

nifti_files = sorted(NIFTI_DIR.glob('*.nii.gz'))
print(f"\n✓ Generated {len(nifti_files)} NIfTI volumes:")
for f in nifti_files:
    print(f"  {f.name}")

In [ ]:
# === Cell 2c: Classify sequences (T1/T2/STIR × sag/ax/cor) ===
import json, pandas as pd, nibabel as nib

def classify(meta):
    desc = (str(meta.get('SeriesDescription', '')) + ' ' + str(meta.get('ProtocolName', ''))).lower()
    te, tr = meta.get('EchoTime'), meta.get('RepetitionTime')
    if 'stir' in desc:                              seq = 'STIR'
    elif 't2' in desc and ('fs' in desc or 'fat' in desc): seq = 'T2_FS'
    elif 't2' in desc:                              seq = 'T2'
    elif 't1' in desc:                              seq = 'T1'
    elif te and tr and te > 60 and tr > 2000:       seq = 'T2'
    elif te and tr and te < 30 and tr < 1000:       seq = 'T1'
    else:                                           seq = 'unknown'
    if 'sag' in desc:                  orient = 'sagittal'
    elif 'ax' in desc or 'tra' in desc: orient = 'axial'
    elif 'cor' in desc:                 orient = 'coronal'
    else:                               orient = 'unknown'
    return seq, orient

sequence_info = []
for nii_path in nifti_files:
    json_path = Path(str(nii_path).replace('.nii.gz', '.json'))
    meta = json.load(open(json_path)) if json_path.exists() else {}
    seq, orient = classify(meta)
    img = nib.load(str(nii_path))
    sequence_info.append({
        'nifti':             str(nii_path),
        'name':              nii_path.name,
        'sequence':          seq,
        'orientation':       orient,
        'shape':             str(img.shape),
        'voxel_mm':          str(tuple(round(s, 2) for s in img.header.get_zooms())),
        'SeriesDescription': meta.get('SeriesDescription', ''),
        'EchoTime':          meta.get('EchoTime'),
        'RepetitionTime':    meta.get('RepetitionTime'),
    })

seq_df = pd.DataFrame(sequence_info)
seq_df.to_csv(INTERMEDIATE / 'sequence_inventory.csv', index=False)
print(seq_df[['name', 'sequence', 'orientation', 'shape']].to_string(index=False))

def pick(seq, orient):
    cands = [s for s in sequence_info if s['sequence'] == seq and s['orientation'] == orient]
    return cands[0]['nifti'] if cands else None

T2_SAG   = pick('T2', 'sagittal')   or pick('T2_FS', 'sagittal')
T1_SAG   = pick('T1', 'sagittal')
STIR_SAG = pick('STIR', 'sagittal') or pick('T2_FS', 'sagittal')
T2_AX    = pick('T2', 'axial')      or pick('T2_FS', 'axial')

print(f"\nPrimary picks:")
print(f"  T2  sag: {Path(T2_SAG).name   if T2_SAG   else 'MISSING'}")
print(f"  T1  sag: {Path(T1_SAG).name   if T1_SAG   else 'MISSING'}")
print(f"  STIR sag: {Path(STIR_SAG).name if STIR_SAG else 'MISSING'}")
print(f"  T2  ax:  {Path(T2_AX).name    if T2_AX    else 'MISSING'}")

# Persist picks
with open(INTERMEDIATE / 'sequence_picks.json', 'w') as f:
    json.dump({'T2_SAG': T2_SAG, 'T1_SAG': T1_SAG, 'STIR_SAG': STIR_SAG, 'T2_AX': T2_AX}, f, indent=2)

## Pipeline A — Anatomical segmentation

SPINEPS → TotalSpineSeg → TotalSegmentator MRI → geometry & radiomics.

In [ ]:
# === Cell 3: SPINEPS — vertebra + posterior elements (v7) ===
# Per official README https://github.com/Hendrik-code/spineps the correct args are:
#   spineps sample -ignore_bids_filter -ignore_inference_compatibility \
#       -i <T2.nii.gz> -model_semantic t2w -model_instance instance
# Weights auto-download on first run. With these args argparse won't call
# format_help, so the Python 3.12 argparse AssertionError bug stays dormant.

SPINEPS_OUT = INTERMEDIATE / 'spineps'
SPINEPS_OUT.mkdir(exist_ok=True)

# SPINEPS writes derivatives next to input. Copy T2 to a working dir so the
# derivatives end up somewhere predictable inside SPINEPS_OUT.
SPINEPS_WORK = SPINEPS_OUT / 'work'
SPINEPS_WORK.mkdir(exist_ok=True)

if T2_SAG is None:
    mark('SPINEPS', 'skipped', 'no T2 sagittal sequence available')
else:
    work_input = SPINEPS_WORK / Path(T2_SAG).name
    if not work_input.exists():
        shutil.copy(T2_SAG, work_input)

    cmd = [
        'spineps', 'sample',
        '-ignore_bids_filter',
        '-ignore_inference_compatibility',
        '-i',              str(work_input),
        '-model_semantic', 't2w',
        '-model_instance', 'instance',
    ]
    print(f"CMD: {' '.join(cmd)}")
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
        print("STDOUT tail:", r.stdout[-1500:])
        if r.returncode != 0:
            print("STDERR tail:", r.stderr[-1500:])
    except subprocess.TimeoutExpired:
        mark('SPINEPS', 'failed', 'timeout 1800s (likely waiting on model download)')
        r = None
    except Exception as e:
        mark('SPINEPS', 'failed', str(e)[:400])
        r = None

    # Detect outputs — SPINEPS writes derivatives folder with vert/spine masks + centroids
    seg_files = list(SPINEPS_OUT.rglob('*vert*.nii.gz'))
    sub_files = list(SPINEPS_OUT.rglob('*spine*.nii.gz'))
    cdt_files = list(SPINEPS_OUT.rglob('*ctd*.json')) + list(SPINEPS_OUT.rglob('*cdt*.json'))
    # Exclude the input file we copied in
    def is_real_output(p):
        return p != work_input and 'work' not in p.parts[-2:]
    seg_files = [p for p in seg_files if is_real_output(p)]
    sub_files = [p for p in sub_files if is_real_output(p)]

    if seg_files or sub_files or cdt_files:
        mark('SPINEPS', 'ok', extra={
            'vertebra_masks': [str(p) for p in seg_files[:5]],
            'subreg_masks':   [str(p) for p in sub_files[:5]],
            'centroid_json':  [str(p) for p in cdt_files[:5]],
        })
    elif r is None or r.returncode == 0:
        mark('SPINEPS', 'failed', 'no output files found (check STDOUT above)')
    else:
        # r exists with non-zero rc — use stderr tail as reason
        reason = (r.stderr or r.stdout).strip().split('\n')[-1][:300]
        mark('SPINEPS', 'failed', reason)


In [ ]:
# === Cell 4: TotalSpineSeg — cord, canal, discs (v7) ===
# v7 fixes:
#   - Input copied to /tmp/totalspineseg_input (NOT inside TSS_OUT) so rglob
#     doesn't false-positive on the input file as if it were an output.
#   - No Python API attempts — totalspineseg has no documented Python API,
#     the v6 candidates were modules (not callables) → 'module' object error.
#   - Real success detection: look for step1_*/step2_*/labels in TSS_OUT.

TSS_OUT = INTERMEDIATE / 'totalspineseg'
TSS_OUT.mkdir(exist_ok=True)

# Input directory OUTSIDE TSS_OUT
TSS_IN = Path('/tmp/totalspineseg_input')
TSS_IN.mkdir(exist_ok=True)
for f in TSS_IN.glob('*'):
    try: f.unlink()
    except: pass

if T2_SAG is None:
    mark('TotalSpineSeg', 'skipped', 'no T2 sagittal')
else:
    shutil.copy(T2_SAG, TSS_IN / Path(T2_SAG).name)
    cmd = ['totalspineseg', str(TSS_IN), str(TSS_OUT), '--iso']
    print(f"CMD: {' '.join(cmd)}")
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=1500)
        print("STDOUT tail:", r.stdout[-1500:])
        if r.returncode != 0:
            print("STDERR tail:", r.stderr[-1500:])
    except subprocess.TimeoutExpired:
        mark('TotalSpineSeg', 'failed', 'timeout 1500s')
        r = None
    except Exception as e:
        mark('TotalSpineSeg', 'failed', str(e)[:400])
        r = None

    # Honest success detection — look for TotalSpineSeg-specific output names.
    # The tool writes step1_output/, step2_output/, etc. and label files.
    real_outputs = []
    for pattern in ['*step*.nii.gz', '*labels*.nii.gz', '*seg*.nii.gz',
                    '*cord*.nii.gz', '*disc*.nii.gz', '*canal*.nii.gz']:
        for p in TSS_OUT.rglob(pattern):
            # Exclude anything still living under the input dir
            if TSS_IN not in p.parents:
                real_outputs.append(p)
    real_outputs = list(dict.fromkeys(real_outputs))  # dedupe preserving order

    if real_outputs:
        mark('TotalSpineSeg', 'ok',
             extra={'segmentations': [str(p) for p in real_outputs[:10]]})
    elif r is None:
        pass  # already marked above
    else:
        reason = (r.stderr or r.stdout).strip().split('\n')[-1][:300] or 'no outputs produced'
        mark('TotalSpineSeg', 'failed', reason)


In [ ]:
# === Cell 5: TotalSegmentator MRI v2 — muscles, ribs ===
TS_OUT = INTERMEDIATE / 'totalsegmentator'
TS_OUT.mkdir(exist_ok=True)

if T2_SAG is None:
    mark('TotalSegmentator', 'skipped', 'no T2 sagittal')
else:
    try:
        from totalsegmentator.python_api import totalsegmentator
        totalsegmentator(
            input=T2_SAG,
            output=str(TS_OUT),
            task='total_mr',
            ml=True,
            roi_subset=None,
            verbose=False,
        )
        outputs = list(TS_OUT.rglob('*.nii.gz'))
        mark('TotalSegmentator', 'ok', extra={'segmentations': [str(p) for p in outputs[:30]]})
    except Exception as e:
        mark('TotalSegmentator', 'failed', str(e)[:400])
        traceback.print_exc()

## Cell 6 — Geometry from segmentations

In [ ]:
# === Cell 6: Geometric metrics — canonical RAS+ orientation (v8) ===
# Previous version picked the smallest dim as "sagittal" and used xs.min()/xs.max()
# as anterior/posterior. That broke whenever the NIfTI was stored in a non-LIA
# orientation — producing absurd wedge angles (54° for C7, -14° for L1).
#
# v8 fix: re-orient with `nib.as_closest_canonical` so that after the transform
# axis 0 = L→R, axis 1 = P→A, axis 2 = I→S. Then the geometry is unambiguous.

import nibabel as nib
import numpy as np
import json

LABEL_NAMES = {**{i: f"C{i}"    for i in range(1, 8)},
               **{i: f"T{i-7}"  for i in range(8, 20)},
               **{i: f"L{i-19}" for i in range(20, 25)}}

global_metrics = {}
per_vertebra = []

# Load SPINEPS centroids
ctd_data = None
ctd_candidates = list(SPINEPS_OUT.rglob('*ctd*.json')) + list(SPINEPS_OUT.rglob('*cdt*.json'))
for ctd_file in ctd_candidates:
    try:
        with open(ctd_file) as f:
            ctd_data = json.load(f)
        if ctd_data:
            print(f"Loaded centroids from {ctd_file.name}")
            break
    except Exception:
        continue

def parse_centroids(ctd):
    out = {}
    if isinstance(ctd, list):
        for item in ctd:
            if isinstance(item, dict) and 'label' in item:
                lbl = int(item['label'])
                out[lbl] = np.array([item.get('X', 0), item.get('Y', 0), item.get('Z', 0)], dtype=float)
    elif isinstance(ctd, dict):
        for k, v in ctd.items():
            try:    lbl = int(k)
            except: continue
            if isinstance(v, dict):
                out[lbl] = np.array([v.get('X', 0), v.get('Y', 0), v.get('Z', 0)], dtype=float)
            elif isinstance(v, (list, tuple)) and len(v) >= 3:
                out[lbl] = np.array(v[:3], dtype=float)
    return out

centroids = parse_centroids(ctd_data) if ctd_data else {}
print(f"Centroids parsed: {len(centroids)}")

# Crude thoracic kyphosis from centroid line
def angle_deg(v1, v2):
    c = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))

if centroids:
    th_labels = sorted(l for l in centroids if 8 <= l <= 19)
    if len(th_labels) >= 3:
        first, last = th_labels[0], th_labels[-1]
        mid = th_labels[len(th_labels)//2]
        v_top    = centroids[mid]   - centroids[first]
        v_bottom = centroids[last]  - centroids[mid]
        global_metrics['thoracic_kyphosis_approx_deg'] = angle_deg(v_top, v_bottom)
    coords = np.stack([centroids[l] for l in th_labels])
    if len(coords) >= 3:
        p0, p1 = coords[0], coords[-1]
        lv = p1 - p0
        lv /= (np.linalg.norm(lv) + 1e-9)
        devs = [np.linalg.norm(p - (p0 + np.dot(p - p0, lv) * lv)) for p in coords]
        global_metrics['max_lateral_deviation_mm'] = float(max(devs))

# Prefer the cleaned vertebra mask over the *raw* output
all_vert = list(SPINEPS_OUT.rglob('*vert_msk*.nii.gz'))
if not all_vert:
    all_vert = list(SPINEPS_OUT.rglob('*vert*.nii.gz'))
seg_files = [p for p in all_vert if 'raw' not in p.name.lower()]
if not seg_files:
    seg_files = all_vert

if seg_files:
    raw_img = nib.load(str(seg_files[0]))
    img = nib.as_closest_canonical(raw_img)
    data = np.asarray(img.get_fdata()).astype(np.int32)
    zooms = img.header.get_zooms()
    print(f"Vertebra mask shape (canonical RAS): {data.shape}, zooms: {tuple(round(z, 2) for z in zooms)}")
    # After canonical: axis 0 = L→R, axis 1 = P→A, axis 2 = I→S
    AP_AXIS, SI_AXIS, LR_AXIS = 1, 2, 0
    ap_mm = zooms[AP_AXIS]
    si_mm = zooms[SI_AXIS]

    def slab_si_extent_vox(slab2d):
        # slab2d shape: (ap_slab, si). Returns SI voxel extent of any-positive column.
        si_present = np.where(slab2d.any(axis=0))[0]
        return int(si_present[-1] - si_present[0] + 1) if len(si_present) >= 2 else 0

    for label_id in np.unique(data):
        if label_id == 0 or label_id > 24:
            continue
        mask = data == label_id
        vox = int(mask.sum())
        if vox < 100:
            continue

        # Project onto sagittal plane: collapse L-R (axis 0)
        sag = mask.any(axis=LR_AXIS)  # shape: (n_ap, n_si)
        if sag.sum() < 20:
            per_vertebra.append({'label_id': int(label_id),
                                  'label_name': LABEL_NAMES.get(int(label_id), f"id_{int(label_id)}"),
                                  'voxel_count': vox})
            continue

        ap_present = np.where(sag.any(axis=1))[0]
        if len(ap_present) < 3:
            per_vertebra.append({'label_id': int(label_id),
                                  'label_name': LABEL_NAMES.get(int(label_id), f"id_{int(label_id)}"),
                                  'voxel_count': vox})
            continue
        ap_min, ap_max = int(ap_present[0]), int(ap_present[-1])

        # Posterior end = LOW ap-index (toward P); Anterior end = HIGH ap-index (toward A)
        slab_thickness = 3
        post_slab = sag[ap_min:ap_min + slab_thickness, :]
        ant_slab  = sag[max(ap_max - slab_thickness + 1, 0):ap_max + 1, :]

        post_h_vox = slab_si_extent_vox(post_slab)
        ant_h_vox  = slab_si_extent_vox(ant_slab)
        ap_width_vox = ap_max - ap_min

        rec = {
            'label_id':       int(label_id),
            'label_name':     LABEL_NAMES.get(int(label_id), f"id_{int(label_id)}"),
            'voxel_count':    vox,
            'ap_width_mm':    round(ap_width_vox * ap_mm, 2),
            'post_height_mm': round(post_h_vox * si_mm, 2),
            'ant_height_mm':  round(ant_h_vox  * si_mm, 2),
        }
        if ant_h_vox > 0 and post_h_vox > 0 and ap_width_vox > 0:
            ap_width_mm  = ap_width_vox * ap_mm
            post_h_mm    = post_h_vox * si_mm
            ant_h_mm     = ant_h_vox  * si_mm
            rec['ant_post_height_ratio'] = round(float(ant_h_mm / post_h_mm), 3)
            # Positive wedge = anterior shorter than posterior (typical Scheuermann pattern)
            rec['wedge_angle_deg'] = round(float(
                np.degrees(np.arctan2(post_h_mm - ant_h_mm, ap_width_mm))), 2)
        per_vertebra.append(rec)

global_metrics['vertebra_count'] = len(per_vertebra)
print(f"\nGlobal metrics: {global_metrics}")
print(f"Per-vertebra records: {len(per_vertebra)}")
if per_vertebra:
    print(f"\nSample wedge angles (should be 0-5° for healthy thoracic):")
    for rec in per_vertebra[:5]:
        print(f"  {rec['label_name']:4s}  wedge={rec.get('wedge_angle_deg', 'n/a'):>6}  "
              f"ratio={rec.get('ant_post_height_ratio', 'n/a'):>5}  "
              f"ant/post={rec.get('ant_height_mm', 'n/a')}/{rec.get('post_height_mm', 'n/a')} mm")

with open(INTERMEDIATE / 'geometry.json', 'w') as f:
    json.dump({'global_metrics': global_metrics, 'per_vertebra': per_vertebra}, f, indent=2)
mark('geometry', 'ok' if per_vertebra else 'partial',
     None if per_vertebra else 'no vertebra masks available')


## Cell 7 — Paraspinal muscle asymmetry (TotalSegmentator outputs)

In [ ]:
# === Cell 7: Muscle CSA + fatty-fraction asymmetry ===
muscle_results = {}
ts_files = list(TS_OUT.rglob('*.nii.gz')) if (TS_OUT := INTERMEDIATE / 'totalsegmentator').exists() else []

muscle_names = ['autochthon', 'erector_spinae', 'multifidus', 'iliocostalis',
                'longissimus', 'spinalis']
# TotalSegmentator MRI v2 uses class names in filenames; pick whatever matches
muscle_files = {p.stem.replace('.nii', ''): p for p in ts_files
                if any(m in p.stem.lower() for m in muscle_names)}

if not muscle_files:
    mark('muscle_analysis', 'skipped', 'no muscle masks found in TotalSegmentator outputs')
else:
    try:
        # Load reference T2 for intensities
        t2_img = nib.load(T2_SAG).get_fdata()
        # Normalize 1-99 percentile to [0,1] for fatty-fraction heuristic
        p1, p99 = np.percentile(t2_img, [1, 99])
        t2_norm = np.clip((t2_img - p1) / (p99 - p1 + 1e-9), 0, 1)

        for name, mpath in muscle_files.items():
            m = nib.load(str(mpath)).get_fdata().astype(bool)
            if m.shape != t2_img.shape:
                continue
            # Split L/R by mid-sagittal plane
            mid_x = m.shape[0] // 2
            left  = m.copy(); left[mid_x:, :, :]  = False
            right = m.copy(); right[:mid_x, :, :] = False

            csa_left  = int(left.sum())
            csa_right = int(right.sum())
            # Fatty fraction proxy: voxels with normalized T2 > 0.6 inside muscle
            fatty_left  = float(((t2_norm > 0.6) & left).sum())  / max(csa_left, 1)
            fatty_right = float(((t2_norm > 0.6) & right).sum()) / max(csa_right, 1)
            asym_pct = 100 * abs(csa_left - csa_right) / max((csa_left + csa_right) / 2, 1)
            muscle_results[name] = {
                'csa_left_voxels':  csa_left,
                'csa_right_voxels': csa_right,
                'asymmetry_pct':    round(asym_pct, 2),
                'fatty_frac_left':  round(fatty_left, 3),
                'fatty_frac_right': round(fatty_right, 3),
            }

        with open(INTERMEDIATE / 'muscle_analysis.json', 'w') as f:
            json.dump(muscle_results, f, indent=2)
        mark('muscle_analysis', 'ok', extra={'muscles': list(muscle_results.keys())})
        for k, v in muscle_results.items():
            print(f"  {k:25s}  asym={v['asymmetry_pct']:5.1f}%  fat L/R={v['fatty_frac_left']:.2f}/{v['fatty_frac_right']:.2f}")
    except Exception as e:
        mark('muscle_analysis', 'failed', str(e)[:400])
        traceback.print_exc()

## Pipeline B — Subtle-finding focus

In [ ]:
# === Cell 8: U2AD — unsupervised T2 anomaly detection ===
# Repo: https://github.com/zhibaishouheilab/U2AD
# Weights may be gated. We attempt clone+import; if it fails, we fall back to a
# simple z-score anomaly heatmap on T2 within the vertebra masks (still useful).
U2AD_OUT = INTERMEDIATE / 'u2ad'
U2AD_OUT.mkdir(exist_ok=True)
U2AD_DIR = EXT_DIR / 'U2AD'
anomaly_results = {'method': None, 'top_findings': []}

def fallback_zscore_anomaly():
    # Simple but effective: per-vertebra T2 z-score, high outlier voxels.
    if T2_SAG is None:
        return None
    t2 = nib.load(T2_SAG).get_fdata()
    if not seg_files:
        return None
    vert = nib.load(str(seg_files[0])).get_fdata().astype(np.int32)
    if vert.shape != t2.shape:
        return None
    findings = []
    for label_id in np.unique(vert):
        if label_id == 0 or label_id > 24:
            continue
        m = vert == label_id
        if m.sum() < 200:
            continue
        vox = t2[m]
        mu, sd = float(vox.mean()), float(vox.std() + 1e-9)
        z = (vox - mu) / sd
        hi = int((z > 3.0).sum())
        if hi > 5:
            findings.append({
                'label_id':       int(label_id),
                'label_name':     LABEL_NAMES.get(int(label_id), f"id_{int(label_id)}"),
                'high_outlier_voxels': hi,
                'mean_intensity': mu,
                'std_intensity':  sd,
                'method':         'per_vertebra_zscore_T2',
            })
    findings.sort(key=lambda x: -x['high_outlier_voxels'])
    return findings[:20]

try:
    if not (U2AD_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/zhibaishouheilab/U2AD.git',
                        str(U2AD_DIR)], check=True, timeout=120)
    sys.path.insert(0, str(U2AD_DIR))
    # U2AD doesn't expose stable Python API → only attempt fallback for now
    raise RuntimeError("U2AD official inference script requires manual weight setup — using fallback")
except Exception as e:
    print(f"[U2AD] {e}\nFalling back to per-vertebra z-score anomaly.")
    findings = fallback_zscore_anomaly()
    if findings is None:
        mark('U2AD', 'skipped', 'no T2 + vertebra masks available')
    else:
        anomaly_results = {'method': 'fallback_zscore_T2', 'top_findings': findings}
        with open(INTERMEDIATE / 'anomaly_findings.json', 'w') as f:
            json.dump(anomaly_results, f, indent=2)
        mark('U2AD', 'fallback_used', extra={'method': 'per_vertebra_zscore_T2',
                                              'top_n': len(findings)})
        print(f"\nTop anomaly hits ({len(findings)}):")
        for f_ in findings[:10]:
            print(f"  {f_['label_name']:5s}  hi-vox={f_['high_outlier_voxels']:4d}")

In [ ]:
# === Cell 9: SpineNetV2 — disc Pfirrmann/Modic (relative ranks) ===
# Repo: https://github.com/rwindsor1/SpineNet (lumbar-trained — we use relative ranks)
SPINENET_OUT = INTERMEDIATE / 'spinenet'
SPINENET_OUT.mkdir(exist_ok=True)
SPINENET_DIR = EXT_DIR / 'SpineNet'

# SpineNetV2 weights are typically gated. We try; on failure we use radiomics-based
# disc-intensity ranking as a transparent substitute (rank discs by T2 mean intensity).
disc_grading = {'method': None, 'discs': []}

def fallback_disc_ranking():
    # Rank discs by mean T2 intensity (lower = more dehydrated).
    # Requires disc masks from TotalSpineSeg.
    if T2_SAG is None:
        return None
    tss_files = list((INTERMEDIATE / 'totalspineseg').rglob('*.nii.gz'))
    disc_mask = None
    for p in tss_files:
        if 'disc' in p.stem.lower() or 'IVD' in p.stem:
            disc_mask = nib.load(str(p)).get_fdata().astype(np.int32)
            break
    if disc_mask is None:
        return None
    t2 = nib.load(T2_SAG).get_fdata()
    if t2.shape != disc_mask.shape:
        return None
    rows = []
    for d_id in np.unique(disc_mask):
        if d_id == 0:
            continue
        m = disc_mask == d_id
        if m.sum() < 50:
            continue
        vals = t2[m]
        rows.append({
            'disc_id':       int(d_id),
            'mean_T2':       float(vals.mean()),
            'voxel_count':   int(m.sum()),
        })
    # Normalize to relative rank 1 (brightest = healthiest) … N (darkest = most dehydrated)
    rows.sort(key=lambda r: -r['mean_T2'])
    for rank, r in enumerate(rows, 1):
        r['relative_dehydration_rank'] = rank
    return rows

try:
    if not (SPINENET_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rwindsor1/SpineNet.git',
                        str(SPINENET_DIR)], check=True, timeout=120)
    raise RuntimeError("SpineNetV2 weights require manual download from authors — using radiomics fallback")
except Exception as e:
    print(f"[SpineNetV2] {e}\nFalling back to T2-intensity disc ranking.")
    discs = fallback_disc_ranking()
    if discs is None:
        mark('SpineNetV2', 'skipped', 'no disc masks (TotalSpineSeg failed or unavailable)')
    else:
        disc_grading = {'method': 'fallback_T2_intensity_rank', 'discs': discs}
        with open(INTERMEDIATE / 'disc_grading.json', 'w') as f:
            json.dump(disc_grading, f, indent=2)
        mark('SpineNetV2', 'fallback_used', extra={'method': 'T2_intensity_rank',
                                                    'n_discs': len(discs)})
        print(f"\nDisc dehydration ranking ({len(discs)} discs):")
        for d in discs[:15]:
            print(f"  disc {d['disc_id']:2d}  mean_T2={d['mean_T2']:.1f}  rank={d['relative_dehydration_rank']}")

In [ ]:
# === Cell 10: Costovertebral joint screen (hybrid: ribs ∩ transverse processes) ===
# No off-the-shelf MRI model exists. We do an intersection of TotalSegmentator
# rib masks with SPINEPS posterior-element masks, dilate, then threshold STIR/T2 for
# bright-signal voxels (edema proxy).
from scipy.ndimage import binary_dilation

cv_results = []
try:
    rib_files = [p for p in list((INTERMEDIATE / 'totalsegmentator').rglob('*.nii.gz'))
                 if 'rib' in p.stem.lower()]
    spineps_sub = list(SPINEPS_OUT.rglob('*seg-spine_msk*.nii.gz')) + \
                  list(SPINEPS_OUT.rglob('*spine*.nii.gz'))

    if not rib_files or not spineps_sub or T2_SAG is None:
        mark('costovertebral_screen', 'skipped',
             'missing rib masks or SPINEPS subreg masks or T2')
    else:
        # Merge rib masks
        rib_mask = None
        for p in rib_files:
            arr = nib.load(str(p)).get_fdata().astype(bool)
            rib_mask = arr if rib_mask is None else rib_mask | arr

        sub = nib.load(str(spineps_sub[0])).get_fdata().astype(np.int32)
        # In SPINEPS subreg scheme: arch / transverse process labels are >40 typically
        post_mask = sub >= 40
        if rib_mask.shape != post_mask.shape:
            mark('costovertebral_screen', 'failed', 'shape mismatch rib vs spine masks')
        else:
            rib_d  = binary_dilation(rib_mask, iterations=2)
            post_d = binary_dilation(post_mask, iterations=2)
            cv_region = rib_d & post_d  # ≈ costotransverse / costovertebral region

            stir = nib.load(STIR_SAG).get_fdata() if STIR_SAG else nib.load(T2_SAG).get_fdata()
            if stir.shape == cv_region.shape:
                p99 = np.percentile(stir[cv_region] if cv_region.any() else stir, 99) if cv_region.any() else 1
                bright = (stir > 0.85 * p99) & cv_region
                # Split L/R
                mid_x = cv_region.shape[0] // 2
                for side, sl in [('L', slice(None, mid_x)), ('R', slice(mid_x, None))]:
                    region = cv_region.copy(); region[sl] = region[sl]  # no-op (clarity)
                    # We use per-side intersection
                    side_region = np.zeros_like(cv_region)
                    side_region[sl] = cv_region[sl]
                    side_bright = bright & side_region
                    cv_results.append({
                        'side':                 side,
                        'region_voxels':        int(side_region.sum()),
                        'bright_voxels':        int(side_bright.sum()),
                        'bright_fraction':      float(side_bright.sum()) / max(int(side_region.sum()), 1),
                    })
                with open(INTERMEDIATE / 'costovertebral.json', 'w') as f:
                    json.dump(cv_results, f, indent=2)
                mark('costovertebral_screen', 'ok')
                for r in cv_results:
                    print(f"  side {r['side']}: region={r['region_voxels']:6d}  bright={r['bright_voxels']:5d}  frac={r['bright_fraction']:.3f}")
except Exception as e:
    mark('costovertebral_screen', 'failed', str(e)[:400])
    traceback.print_exc()

## Cell 11 — Radiomics texture features

In [ ]:
# === Cell 11: Texture features (pyradiomics OR scikit-image fallback) ===
radiomics_results = []
use_skimage_fallback = False

try:
    from radiomics import featureextractor
    import SimpleITK as sitk
    import logging
    logging.getLogger('radiomics').setLevel(logging.ERROR)
    print("Using pyradiomics for texture features")
except ImportError:
    print("pyradiomics not available — falling back to scikit-image (first-order + GLCM)")
    use_skimage_fallback = True
    from skimage.feature import graycomatrix, graycoprops
    from scipy.stats import skew, kurtosis
    import SimpleITK as sitk

try:
    if T2_SAG is None or not seg_files:
        mark('pyradiomics', 'skipped', 'no T2 or vertebra masks')
    elif use_skimage_fallback:
        t2 = nib.load(T2_SAG).get_fdata()
        vert = nib.load(str(seg_files[0])).get_fdata().astype(np.int32)
        if t2.shape != vert.shape:
            mark('pyradiomics', 'failed', f'shape mismatch {t2.shape} vs {vert.shape}')
        else:
            # Quantize T2 to 32 levels for GLCM
            t2_p1, t2_p99 = np.percentile(t2, [1, 99])
            t2_q = np.clip((t2 - t2_p1) / (t2_p99 - t2_p1 + 1e-9), 0, 1)
            t2_q = (t2_q * 31).astype(np.uint8)
            for label_id in np.unique(vert):
                if label_id == 0 or label_id > 24:
                    continue
                m = vert == label_id
                if m.sum() < 100:
                    continue
                vox = t2[m]
                # First-order
                feats = {
                    'mean':       float(vox.mean()),
                    'std':        float(vox.std()),
                    'min':        float(vox.min()),
                    'max':        float(vox.max()),
                    'median':     float(np.median(vox)),
                    'p10':        float(np.percentile(vox, 10)),
                    'p90':        float(np.percentile(vox, 90)),
                    'skewness':   float(skew(vox)),
                    'kurtosis':   float(kurtosis(vox)),
                    'energy':     float(np.sum(vox.astype(np.float64) ** 2) / max(vox.size, 1)),
                    'voxel_count': int(m.sum()),
                }
                # GLCM on mid-sagittal slice intersection
                sag_axis = int(np.argmin(vert.shape))
                mid = vert.shape[sag_axis] // 2
                mask2d = np.take(m, mid, axis=sag_axis)
                t2q_2d = np.take(t2_q, mid, axis=sag_axis)
                if mask2d.sum() > 50:
                    patch = t2q_2d.copy()
                    patch[~mask2d] = 0
                    try:
                        glcm = graycomatrix(patch, distances=[1], angles=[0, np.pi/2],
                                            levels=32, symmetric=True, normed=True)
                        feats['glcm_contrast']      = float(graycoprops(glcm, 'contrast').mean())
                        feats['glcm_homogeneity']   = float(graycoprops(glcm, 'homogeneity').mean())
                        feats['glcm_correlation']   = float(graycoprops(glcm, 'correlation').mean())
                        feats['glcm_energy']        = float(graycoprops(glcm, 'energy').mean())
                        feats['glcm_dissimilarity'] = float(graycoprops(glcm, 'dissimilarity').mean())
                    except Exception:
                        pass
                radiomics_results.append({
                    'label_id':   int(label_id),
                    'label_name': LABEL_NAMES.get(int(label_id), f"id_{int(label_id)}"),
                    'features':   feats,
                })
            mark('pyradiomics', 'fallback_used',
                 extra={'method': 'skimage_GLCM_plus_firstorder', 'n_vertebrae': len(radiomics_results)})
    else:
        # Standard pyradiomics path
        extractor = featureextractor.RadiomicsFeatureExtractor()
        for cls in ['firstorder', 'glcm', 'glrlm', 'shape']:
            extractor.enableFeatureClassByName(cls)
        t2_sitk = sitk.ReadImage(T2_SAG)
        seg_sitk_full = sitk.ReadImage(str(seg_files[0]))
        seg_arr = sitk.GetArrayFromImage(seg_sitk_full)
        for label_id in np.unique(seg_arr):
            if label_id == 0 or label_id > 24:
                continue
            mask_arr = (seg_arr == label_id).astype(np.uint8)
            if mask_arr.sum() < 100:
                continue
            mask_img = sitk.GetImageFromArray(mask_arr)
            mask_img.CopyInformation(seg_sitk_full)
            try:
                feats = extractor.execute(t2_sitk, mask_img, label=1)
                feat_dict = {k: float(v) for k, v in feats.items()
                             if k.startswith(('original_firstorder', 'original_glcm',
                                              'original_glrlm', 'original_shape'))
                             and isinstance(v, (int, float, np.floating))}
                radiomics_results.append({
                    'label_id':   int(label_id),
                    'label_name': LABEL_NAMES.get(int(label_id), f"id_{int(label_id)}"),
                    'features':   feat_dict,
                })
            except Exception as ex:
                print(f"  [skip label {label_id}] {ex}")
        mark('pyradiomics', 'ok', extra={'n_vertebrae': len(radiomics_results)})

    if radiomics_results:
        with open(INTERMEDIATE / 'radiomics.json', 'w') as f:
            json.dump(radiomics_results, f, indent=2)
        print(f"Extracted features for {len(radiomics_results)} vertebrae")
except Exception as e:
    mark('pyradiomics', 'failed', str(e)[:400])
    traceback.print_exc()


## Cell 12 — Cross-tool agreement (sanity check)

In [ ]:
# === Cell 12: Cross-tool agreement (SPINEPS cord vs TotalSpineSeg cord) ===
agreement = {}
try:
    sp_cord = None
    for p in SPINEPS_OUT.rglob('*.nii.gz'):
        if 'cord' in p.stem.lower() or 'spinalcord' in p.stem.lower():
            sp_cord = nib.load(str(p)).get_fdata().astype(bool)
            break
    tss_cord = None
    for p in (INTERMEDIATE / 'totalspineseg').rglob('*.nii.gz'):
        if 'cord' in p.stem.lower() or 'spinalcord' in p.stem.lower():
            tss_cord = nib.load(str(p)).get_fdata().astype(bool)
            break

    if sp_cord is None or tss_cord is None:
        mark('cross_tool_agreement', 'skipped', 'cord masks not found from both tools')
    elif sp_cord.shape != tss_cord.shape:
        mark('cross_tool_agreement', 'skipped', f'shape mismatch {sp_cord.shape} vs {tss_cord.shape}')
    else:
        inter = (sp_cord & tss_cord).sum()
        denom = sp_cord.sum() + tss_cord.sum()
        dice = 2.0 * inter / denom if denom else 0.0
        agreement = {
            'spineps_cord_voxels':       int(sp_cord.sum()),
            'totalspineseg_cord_voxels': int(tss_cord.sum()),
            'intersection_voxels':       int(inter),
            'dice_score':                round(float(dice), 3),
            'warning':                   'low_agreement' if dice < 0.7 else None,
        }
        with open(INTERMEDIATE / 'cross_tool_agreement.json', 'w') as f:
            json.dump(agreement, f, indent=2)
        mark('cross_tool_agreement', 'ok', extra={'dice': agreement['dice_score']})
        print(f"  Dice (SPINEPS vs TotalSpineSeg cord): {dice:.3f}")
        if dice < 0.7:
            print("  ⚠ Low agreement — manual review recommended")
except Exception as e:
    mark('cross_tool_agreement', 'failed', str(e)[:400])
    traceback.print_exc()

## Cell 13 — Final aggregation

In [ ]:
# === Cell 13: Build findings.json + findings.csv ===
import pandas as pd

def load_json(p, default=None):
    try:
        with open(p) as f: return json.load(f)
    except Exception:
        return default

geom            = load_json(INTERMEDIATE / 'geometry.json',          {'global_metrics': {}, 'per_vertebra': []})
muscle          = load_json(INTERMEDIATE / 'muscle_analysis.json',   {})
anomaly         = load_json(INTERMEDIATE / 'anomaly_findings.json',  {'method': None, 'top_findings': []})
disc            = load_json(INTERMEDIATE / 'disc_grading.json',      {'method': None, 'discs': []})
cv              = load_json(INTERMEDIATE / 'costovertebral.json',    [])
radiomics_data  = load_json(INTERMEDIATE / 'radiomics.json',         [])
agreement_data  = load_json(INTERMEDIATE / 'cross_tool_agreement.json', {})

# Index radiomics by label
rad_by_label = {r['label_id']: r['features'] for r in radiomics_data}

# Merge per-vertebra info
per_vert_merged = []
for v in geom.get('per_vertebra', []):
    lid = v['label_id']
    rec = dict(v)
    rec['radiomics']     = rad_by_label.get(lid, {})
    # Attach anomaly score if present
    for af in anomaly.get('top_findings', []):
        if af['label_id'] == lid:
            rec['anomaly_T2'] = {
                'high_outlier_voxels': af.get('high_outlier_voxels'),
                'method':              af.get('method'),
            }
            break
    per_vert_merged.append(rec)

findings = {
    'patient_id':       'anon',
    'pipeline_version': '2026-05-21',
    'tool_status':      STATUS['tools'],
    'global_metrics':   geom.get('global_metrics', {}),
    'per_vertebra':     per_vert_merged,
    'per_disc':         disc.get('discs', []),
    'disc_grading_method': disc.get('method'),
    'paraspinal_muscles': muscle,
    'costovertebral':   cv,
    'top_anomalies':    anomaly.get('top_findings', []),
    'anomaly_method':   anomaly.get('method'),
    'cross_tool_agreement': agreement_data,
    'sequences_used':   {
        'T2_SAG':   T2_SAG,
        'T1_SAG':   T1_SAG,
        'STIR_SAG': STIR_SAG,
        'T2_AX':    T2_AX,
    },
}

OUT_JSON = RESULTS_DIR / 'findings.json'
with open(OUT_JSON, 'w') as f:
    json.dump(findings, f, indent=2, default=str)
print(f"✓ Saved {OUT_JSON} ({OUT_JSON.stat().st_size / 1024:.1f} KB)")

# Flat CSV (per-vertebra row)
csv_rows = []
for v in per_vert_merged:
    row = {
        'label_name':     v.get('label_name'),
        'label_id':       v.get('label_id'),
        'voxel_count':    v.get('voxel_count'),
        'wedge_angle_deg':         v.get('wedge_angle_deg'),
        'ant_post_height_ratio':   v.get('ant_post_height_ratio'),
        'anomaly_T2_voxels':       (v.get('anomaly_T2') or {}).get('high_outlier_voxels'),
    }
    for fk, fv in (v.get('radiomics') or {}).items():
        row[fk] = fv
    csv_rows.append(row)

if csv_rows:
    OUT_CSV = RESULTS_DIR / 'findings.csv'
    pd.DataFrame(csv_rows).to_csv(OUT_CSV, index=False)
    print(f"✓ Saved {OUT_CSV}")
else:
    print("[no per-vertebra rows to write to CSV]")

# Brief summary
print("\n" + "=" * 60)
print("TOOL STATUS SUMMARY")
print("=" * 60)
for tool, s in STATUS['tools'].items():
    icon = {'ok': '✓', 'failed': '✗', 'skipped': '○',
            'fallback_used': '◐', 'partial': '◐'}.get(s['status'], '?')
    print(f"  {icon} {tool:25s} {s['status']:15s}" + (f" — {s['reason']}" if s.get('reason') else ""))
print(f"\nGlobal metrics:")
for k, v in findings['global_metrics'].items():
    print(f"  {k}: {v}")
print(f"\nVertebrae analyzed: {len(per_vert_merged)}")
print(f"Anomaly findings:   {len(findings['top_anomalies'])}")
print(f"Disc records:       {len(findings['per_disc'])}")
print(f"\nFinal results in: {RESULTS_DIR}/")
print("  - findings.json   (full structured output)")
print("  - findings.csv    (flat per-vertebra table)")
print("  - intermediate/   (per-tool raw outputs)")

## Next steps after running

1. Download `results/findings.json` from Colab → commit to repo at `results/findings.json`.
2. Inspect `STATUS` block at top of JSON — note any `failed` / `skipped` tools.
3. **Tools likely needing manual setup**:
   - **SPINEPS** — if it failed on first run, weights may have not auto-downloaded; check the SPINEPS repo for `--download_weights` flag or HuggingFace links in their README.
   - **U2AD** — official inference requires custom training/weights; for now we use a transparent z-score fallback. If you want full U2AD, expect ~1–2 hours of setup.
   - **SpineNetV2** — weights are gated; request from `rwindsor1/SpineNet`. Fallback (T2 intensity rank) is informative for relative comparison.
4. Bring the findings back into this conversation and I'll help interpret the anomaly hits and rank likely diagnostic targets (Scheuermann grade, facet hyperintensity, costovertebral edema, multifidus asymmetry).